In [ ]:
%load_ext autoreload
%autoreload 2

## S0: Temporal Persistence
Oracle δ vs LS δ, NMSE from reuse.

In [8]:
import sys; sys.path.insert(0, "../..")  # project root
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.experiments.pipeline import load_all_ues, run_s0, run_s1, run_s2, run_s3, run_s4, run_s5, run_s7, SPEED_LABELS
plt.style.use("seaborn-v0_8-whitegrid")

PRESET = "munich_elaa_m_1k_15g"
MAX_SNAPSHOTS = 4000
UE_PER_SPEED = 2


In [ ]:

# Phase 1: Load UEs (slow, ~3min). Stays in RAM for all experiments.
ue_data = load_all_ues(PRESET, max_snapshots=MAX_SNAPSHOTS, ue_per_speed=UE_PER_SPEED)

In [ ]:
s0 = run_s0(ue_data)

per_ue = s0["per_ue"]
summary = s0["summary"]
speeds = np.array([u["speed"] for u in per_ue])
deltas_oracle = np.array([u["median_delta"] for u in per_ue])
deltas_ls = np.array([u.get("median_delta_ls", u["median_delta"]) for u in per_ue])
nmses = np.array([u["nmse_reuse_db"] for u in per_ue])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1a: Speed vs δ
ax = axes[0]
for i, u in enumerate(per_ue):
    spd, d_o, d_l = u["speed"], u["median_delta"], u.get("median_delta_ls", u["median_delta"])
    ax.scatter(spd - 0.15, d_o, c="tab:blue", s=80, edgecolors="k", linewidths=0.5, marker="o",
               label="Oracle δ" if i == 0 else "", zorder=5)
    ax.scatter(spd + 0.15, d_l, c="tab:orange", s=80, edgecolors="k", linewidths=0.5, marker="s",
               label="LS δ (20dB)" if i == 0 else "", zorder=5)
    ax.annotate(f"UE{u['uid']}", (spd + 0.2, max(d_o, d_l)), textcoords="offset points", xytext=(5, 3), fontsize=7)
ax.axhline(summary["static_median_delta"], color="red", ls="--", alpha=0.5,
           label=f"static noise floor ({summary['static_median_delta']:.4f})")
ax.set_xlabel("Speed (m/s)"); ax.set_ylabel("Median δ"); ax.set_xlim(-0.5, 9.5)
ax.set_title("Speed → Channel Variation"); ax.legend(fontsize=7, loc="upper left")

# 1b: Grouped bar
ax = axes[1]
cats = {"static": 0.0, "ped\n(1m/s)": 1.0, "veh\n(8.3m/s)": 8.3}
x_pos = np.arange(len(cats)); w = 0.35
for j, (arr, lbl, c) in enumerate([(deltas_oracle, "Oracle δ", "tab:blue"), (deltas_ls, "LS δ (20dB)", "tab:orange")]):
    vals = [np.median(arr[np.abs(speeds - s) < 0.1]) if (np.abs(speeds - s) < 0.1).any() else 0 for _, s in cats.items()]
    ax.bar(x_pos + (j - 0.5) * w, vals, w, color=c, alpha=0.8, label=lbl)
ax.set_xticks(x_pos); ax.set_xticklabels(cats.keys()); ax.set_ylabel("Median δ")
ax.set_title("δ by Mobility: Oracle vs LS"); ax.legend(fontsize=8)

# 1c: NMSE
ax = axes[2]
for i, (lbl, s) in enumerate({"static": 0.0, "ped": 1.0, "veh": 8.3}.items()):
    mask = np.abs(speeds - s) < 0.1; v = nmses[mask]; v = v[~np.isnan(v)]
    if len(v) > 0: ax.bar(i, np.median(v), color=["blue","green","red"][i], alpha=0.7, label=f"{lbl}: {np.median(v):.1f}dB")
ax.axhline(-10, color="gray", ls=":", alpha=0.5, label="-10dB")
ax.set_xticks(range(3)); ax.set_xticklabels(["static","ped","veh"]); ax.set_ylabel("NMSE(reuse) [dB]")
ax.set_title("Quality Loss from Reuse"); ax.legend(fontsize=8)

fig.suptitle(f"S0: {PRESET} ({summary['n_ues']} UEs, {MAX_SNAPSHOTS} snaps)", fontsize=12)
plt.tight_layout(); plt.show()
print(f"GO/NO-GO: {'GO ✓' if summary['median_delta'] <= 0.5 else 'NO-GO ✗'} (median δ = {summary['median_delta']:.4f})")

## S1: Does Scheduling Work? (Single-τ)
Core question: at τ=0.2, how does adaptive scheduling compare to always-full-CE and always-skip?

In [7]:
from src.experiments.pipeline import run_s1
s1 = run_s1(ue_data, tau=0.2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

uids = [u["uid"] for u in s1["per_ue"]]
labels = [f"UE{u['uid']}\n({SPEED_LABELS.get(round(u['speed'],1),'?')})" 
          if 'SPEED_LABELS' in dir() else f"UE{u['uid']}" for u in s1["per_ue"]]
# Fix: define labels properly
labels = []
for u in s1["per_ue"]:
    spd = u["speed"]
    lbl = "static" if spd < 0.1 else ("ped" if spd < 5 else "veh")
    labels.append(f"UE{u['uid']}\n({lbl})")

nmse_full = [u["nmse_full_db"] for u in s1["per_ue"]]
nmse_skip = [u["nmse_skip_db"] for u in s1["per_ue"]]
nmse_adapt = [u["nmse_adaptive_db"] for u in s1["per_ue"]]

# 1a: NMSE comparison bar chart
ax = axes[0]
x = np.arange(len(labels))
w = 0.25
ax.bar(x - w, nmse_full, w, label="Always Full CE", color="tab:blue", alpha=0.8)
ax.bar(x, nmse_adapt, w, label=f"Adaptive (τ={s1['tau']})", color="tab:green", alpha=0.8)
ax.bar(x + w, nmse_skip, w, label="Always Skip", color="tab:red", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("NMSE (dB)"); ax.set_title("S1: NMSE by Strategy")
ax.axhline(-10, color="gray", ls=":", alpha=0.5, label="-10dB")
ax.legend(fontsize=8)

# 1b: Tier distribution (skip/delta/full %)
ax = axes[1]
skip_r = [u["skip_rate"] for u in s1["per_ue"]]
delta_r = [u["delta_rate"] for u in s1["per_ue"]]
full_r = [u["full_rate"] for u in s1["per_ue"]]
ax.bar(x, skip_r, label="Skip (T0)", color="tab:green", alpha=0.8)
ax.bar(x, delta_r, bottom=skip_r, label="Delta (T1)", color="tab:orange", alpha=0.8)
ax.bar(x, full_r, bottom=[s+d for s,d in zip(skip_r, delta_r)], label="Full (T2)", color="tab:red", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Fraction"); ax.set_title(f"S1: Tier Distribution (τ={s1['tau']})")
ax.legend(fontsize=8)

fig.suptitle(f"S1: {ue_data['preset']} — Does Scheduling Work?", fontsize=12)
plt.tight_layout(); plt.show()

# Summary
print(f"\nS1 Summary (τ={s1['tau']}, SNR={s1['snr_db']}dB):")
for u in s1["per_ue"]:
    lbl = "static" if u["speed"]<0.1 else ("ped" if u["speed"]<5 else "veh")
    gain = u["nmse_adaptive_db"] - u["nmse_full_db"]
    print(f"  UE{u['uid']}({lbl}): Full={u['nmse_full_db']:.1f}dB, Adaptive={u['nmse_adaptive_db']:.1f}dB, "
          f"Skip={u['nmse_skip_db']:.1f}dB | SR={u['skip_rate']:.0%} | gap={gain:+.1f}dB")

S1: scheduling (τ=0.2):   0%|          | 0/6 [00:00<?, ?ue/s]


KeyboardInterrupt



## S2: Threshold Sweep → Pareto Front
τ sweep: skip rate vs NMSE tradeoff.

In [ ]:
s2 = run_s2(ue_data)

sweep = s2["sweep"]
taus = [s["tau"] for s in sweep]
nmses_s2 = [s["avg_nmse_db"] for s in sweep]
srs = [s["skip_rate"] for s in sweep]
crs = [s["computation_ratio"] for s in sweep]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(srs, nmses_s2, "b-o", ms=4)
ax.set_xlabel("Skip Rate"); ax.set_ylabel("NMSE (dB)"); ax.set_title("S2: Skip Rate vs NMSE")
ax.axhline(-10, color="gray", ls=":", label="-10dB"); ax.legend(); ax.invert_xaxis()

ax = axes[1]
ax.plot(crs, nmses_s2, "r-s", ms=4)
ax.set_xlabel("Computation Ratio"); ax.set_ylabel("NMSE (dB)"); ax.set_title("S2: Pareto Front")
ax.axhline(-10, color="gray", ls=":", label="-10dB"); ax.legend()

fig.suptitle(f"S2: {PRESET}", fontsize=12); plt.tight_layout(); plt.show()

sweet = [s for s in sweep if s["avg_nmse_db"] <= -10]
if sweet:
    best = min(sweet, key=lambda s: s["computation_ratio"])
    print(f"Sweet spot: τ={best['tau']:.2f}, SR={best['skip_rate']:.0%}, CR={best['computation_ratio']:.3f}, NMSE={best['avg_nmse_db']:.1f}dB")

## S4: Delta Update Ablation
Compare T1 modes: pure skip vs EMA vs LS-delta at τ=0.2.

In [ ]:
s4 = run_s4(ue_data)

modes = list(s4.keys())
speed_cats = ["static", "ped", "veh"]
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(speed_cats)); w = 0.8 / len(modes)
for j, mode in enumerate(modes):
    vals = [s4[mode].get(sc, {}).get("nmse_db", 0) for sc in speed_cats]
    ax.bar(x + j * w, vals, w, label=mode, alpha=0.8)
ax.set_xticks(x + w * len(modes) / 2); ax.set_xticklabels(speed_cats)
ax.set_ylabel("NMSE (dB)"); ax.set_title("S4: Delta Update Ablation (τ=0.2)")
ax.legend(fontsize=7, ncol=3); ax.axhline(-10, color="gray", ls=":", alpha=0.5)
plt.tight_layout(); plt.show()

for sc in speed_cats:
    best = min(modes, key=lambda m: s4[m].get(sc, {}).get("nmse_db", 0))
    print(f"  {sc}: best={best} → {s4[best].get(sc, {}).get('nmse_db', float('nan')):.1f}dB")

## S5: Beamforming Rate Impact
Rate loss from stale channel estimates across SNR × τ.

In [ ]:
s5 = run_s5(ue_data)

snr_keys = sorted(s5.keys()); tau_keys = sorted(s5[snr_keys[0]].keys())
rpr_mat = np.array([[s5[sk][tk]["avg_rpr"] for tk in tau_keys] for sk in snr_keys])
rl5_mat = np.array([[s5[sk][tk]["frac_above_5pct"] for tk in tau_keys] for sk in snr_keys])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, cmap, vr, title in [
    (axes[0], rpr_mat, "RdYlGn", (0.8, 1.0), "RPR"),
    (axes[1], rl5_mat, "RdYlGn_r", (0, 0.5), "P(rate_loss > 5%)"),
]:
    im = ax.imshow(mat, aspect="auto", cmap=cmap, vmin=vr[0], vmax=vr[1])
    ax.set_xticks(range(len(tau_keys))); ax.set_xticklabels([t.replace("tau","τ=") for t in tau_keys], fontsize=8)
    ax.set_yticks(range(len(snr_keys))); ax.set_yticklabels([s.replace("snr","")+"dB" for s in snr_keys])
    ax.set_xlabel("τ"); ax.set_ylabel("SNR"); ax.set_title(f"S5: {title}")
    for i in range(len(snr_keys)):
        for j in range(len(tau_keys)):
            fmt = f"{mat[i,j]:.3f}" if "RPR" in title else f"{mat[i,j]:.0%}"
            ax.text(j, i, fmt, ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax)
fig.suptitle(f"S5: {PRESET}", fontsize=12); plt.tight_layout(); plt.show()

print("Sweet spots (RPR>0.95, P(rl>5%)<10%):")
for i, sk in enumerate(snr_keys):
    for j, tk in enumerate(tau_keys):
        if rpr_mat[i,j] > 0.95 and rl5_mat[i,j] < 0.10:
            print(f"  ✓ {sk} {tk}: RPR={rpr_mat[i,j]:.3f}")

## S3: Distance-Dependent Threshold
UEs classified by Rayleigh distance zones.

In [ ]:
s3 = run_s3(ue_data)
r_ray = s3["r_rayleigh"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for u in s3["per_ue"]:
    c = {"NF": "red", "Transition": "orange", "FF": "blue"}.get(u["zone"], "gray")
    s0_match = [x for x in s0["per_ue"] if x["uid"] == u["uid"]]
    d = s0_match[0]["median_delta"] if s0_match else 0
    ax.scatter(u["dist"], d, c=c, s=80, edgecolors="k", linewidths=0.5, zorder=5)
    ax.annotate(f"UE{u['uid']}", (u["dist"], d), textcoords="offset points", xytext=(5, 3), fontsize=7)
ax.axvline(r_ray, color="red", ls="--", alpha=0.5, label=f"R_Ray={r_ray:.1f}m")
ax.axvline(3 * r_ray, color="orange", ls="--", alpha=0.5, label=f"3×R_Ray={3*r_ray:.1f}m")
ax.set_xlabel("Distance (m)"); ax.set_ylabel("Median δ_oracle"); ax.set_title("S3: δ vs Distance"); ax.legend(fontsize=8)

ax = axes[1]
zones = ["NF", "Transition", "FF"]
taus_z = [s3["zones"].get(z, {}).get("optimal_tau", 0) or 0 for z in zones]
counts = [s3["zones"].get(z, {}).get("count", 0) for z in zones]
ax.bar(range(3), taus_z, color=["red", "orange", "blue"], alpha=0.7)
for i, (t, c) in enumerate(zip(taus_z, counts)):
    ax.text(i, t + 0.01, f"τ*={t:.2f}\n(n={c})", ha="center", fontsize=9)
ax.set_xticks(range(3)); ax.set_xticklabels(zones); ax.set_ylabel("Optimal τ*"); ax.set_title("S3: Distance-Dependent τ*")

fig.suptitle(f"S3: {PRESET} (R_Ray={r_ray:.1f}m)", fontsize=12); plt.tight_layout(); plt.show()

## S7: System Overhead & Effective Throughput
Scheduling overhead breakdown + system capacity gain.

In [ ]:
s7 = run_s7(ue_data, s2_result=s2)

oh = s7["overhead"]; eff = s7["effective_throughput"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.bar(["LS\n(Y/X)", "Monitor\n(δ calc)", "Full CE\n(LMMSE)"],
              [oh["t_ls_ms"], oh["t_monitor_ms"], oh["t_full_ce_ms"]],
              color=["lightblue", "orange", "red"], alpha=0.8)
ax.set_ylabel("Time (ms)"); ax.set_yscale("log"); ax.set_title("S7: Component Times")
for b, t in zip(bars, [oh["t_ls_ms"], oh["t_monitor_ms"], oh["t_full_ce_ms"]]):
    ax.text(b.get_x() + b.get_width()/2, t * 1.5, f"{t:.2e}ms", ha="center", fontsize=8)
ax.text(0.95, 0.95, f"Overhead: {oh['overhead_ratio']:.4%}\nMemory: {oh['memory_bytes']/1024:.0f}KB",
        transform=ax.transAxes, va="top", ha="right", fontsize=9, bbox=dict(boxstyle="round", fc="wheat", alpha=0.8))

ax = axes[1]
vals = [eff["per_ue_rate_factor"], eff["capacity_multiplier"], eff["system_throughput_gain"]]
bars = ax.bar(["Per-UE\nRate", "Capacity\nMultiplier", "System\nThroughput"], vals, color=["green","blue","purple"], alpha=0.7)
ax.axhline(1.0, color="gray", ls="--", alpha=0.5)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.1, f"{v:.2f}x", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Multiplier"); ax.set_title("S7: System Impact")

fig.suptitle(f"S7: {PRESET}", fontsize=12); plt.tight_layout(); plt.show()
print(f"Overhead: {oh['overhead_ratio']:.4%} | Capacity: {eff['capacity_multiplier']:.1f}x | Throughput: {eff['system_throughput_gain']:.1f}x")